# EDA - e-commerce (Olist)

Goals:
- Load each dataset and inspect schema
- Check missing values and duplicates
- Validate key uniqueness and join coverage
- Review date ranges and numeric outliers


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
if not (ROOT / 'e-commerce').exists() and (ROOT.parent / 'e-commerce').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'e-commerce'


In [3]:
orders = pd.read_csv(
    DATA_DIR / 'olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
)

customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
order_items = pd.read_csv(
    DATA_DIR / 'olist_order_items_dataset.csv',
    parse_dates=['shipping_limit_date'],
)
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(
    DATA_DIR / 'olist_order_reviews_dataset.csv',
    parse_dates=['review_creation_date', 'review_answer_timestamp'],
)
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
geolocation = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
category_translation = pd.read_csv(
    DATA_DIR / 'product_category_name_translation.csv',
    encoding='utf-8-sig',
)


FileNotFoundError: [Errno 2] No such file or directory: 'e-commerce/olist_orders_dataset.csv'

In [ ]:
def profile_df(df, name, id_cols=None):
    print(f'\n{name}')
    print('shape:', df.shape)
    print('dtypes:')
    print(df.dtypes)
    missing = df.isna().sum().sort_values(ascending=False)
    print('missing (top):')
    print(missing[missing > 0].head(20))
    dup = df.duplicated().sum()
    print('duplicate rows:', dup)
    if id_cols:
        for col in id_cols:
            nunique = df[col].nunique()
            print(f'unique {col}:', nunique)


In [ ]:
profile_df(customers, 'customers', id_cols=['customer_id', 'customer_unique_id'])
profile_df(orders, 'orders', id_cols=['order_id', 'customer_id'])
profile_df(order_items, 'order_items', id_cols=['order_id', 'product_id', 'seller_id'])
profile_df(payments, 'payments', id_cols=['order_id'])
profile_df(reviews, 'reviews', id_cols=['review_id', 'order_id'])
profile_df(products, 'products', id_cols=['product_id'])
profile_df(sellers, 'sellers', id_cols=['seller_id'])
profile_df(geolocation, 'geolocation')


In [ ]:
def join_coverage(left, right, key):
    left_keys = left[key].dropna().unique()
    right_keys = right[key].dropna().unique()
    left_in_right = np.isin(left_keys, right_keys).mean()
    right_in_left = np.isin(right_keys, left_keys).mean()
    return {
        'left_unique': len(left_keys),
        'right_unique': len(right_keys),
        'left_in_right_ratio': left_in_right,
        'right_in_left_ratio': right_in_left,
    }

print('orders vs payments:', join_coverage(orders, payments, 'order_id'))
print('orders vs items:', join_coverage(orders, order_items, 'order_id'))
print('orders vs reviews:', join_coverage(orders, reviews, 'order_id'))
print('products vs items:', join_coverage(products, order_items, 'product_id'))
print('sellers vs items:', join_coverage(sellers, order_items, 'seller_id'))


In [ ]:
print(orders[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']].agg(['min', 'max']))


In [ ]:
print(order_items[['price', 'freight_value']].quantile([0.5, 0.9, 0.99]))
print(products[['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].describe())


In [ ]:
missing_categories = products['product_category_name'].isna().sum()
print('missing product_category_name:', missing_categories)
merged = products.merge(category_translation, on='product_category_name', how='left')
print('missing translation rows:', merged['product_category_name_english'].isna().sum())
